In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [3]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [4]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
                                      if pd.notna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

    # eliminar , al final del ejercicio
    df.ejercicio = df.ejercicio.str.strip(",")

    # create folder for clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)

    return df

Arreglar ejercicios de Ponencia

In [5]:
def fix_ponencia(row):
    if "PONENCIA" in row["info_ejercicio"]:
        row["convocatoria"] = "Ponencia"
        pos_ejercicio = row["info_ejercicio"].find("EJERCICIO")
        row["ejercicio"] = row["info_ejercicio"][pos_ejercicio:].title()
    return row

In [6]:
df = df.apply(fix_ponencia, axis = 1)

In [7]:
df[df.ejercicio.isna()]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,Mates II,2017,NaN,NaN,Matrices Y Determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,Mates II - Temas/matrices_determinantes/2017 -...,png/png_Mates II_2017 - Matrices Y Determinant...,NaN


In [8]:
df.loc[2556, "info_ejercicio"]

'MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE RUC CIC II O Ó 3 .N O PCIÓN B.'

In [9]:
df.loc[2556, ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [10]:
df.sample()

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
1246,Mates CCSS,Mates CCSS,2017,Junio,Ejercicio 3. Opción A,Probabilidad,SOCIALES II. 2017. JUNIO. EJERCICIO 3. OPCIÓN A,Se sabe que el 90% de los alumnos de un centro...,Mates CCSS - Temas/probabilidad/2017 - Probabi...,png/png_Mates CCSS_2017 - Probabilidad/Mates C...,NaN


In [11]:
df = clean_dataframe(df)

Tras la limpieza completa, creo el ID

In [12]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [13]:
# create id
df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
# rearrange columns
df = df[["id"] + list(df.columns[:-2])]

Exportar el dataframe limpio

In [14]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [15]:
df.sample(5)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
4041,quimica_formulacion_2013_reserva_2_ejercicio_1...,Química,Química,2013,Reserva 2,"Ejercicio 1, opción B",Formulación,QUÍMICA. 2013. RESERVA 2. EJERCICIO 1. OPCIÓN B,Formule o nombre los siguientes compuestos: a)...,Química - Temas/formulación/2013 - Formulaci...,png/png_Química_2013 - Formulación/Química_f...
1789,mates_ii_funciones_2019_reserva_4_ejercicio_1_...,Mates II,Mates II,2019,Reserva 4,"Ejercicio 1, opción A",Funciones,MATEMÁTICAS II. 2019. RESERVA 4. EJERCICIO 1. ...,"Dada la función f :0,2 , definida por f(x)...",Mates II - Temas/funciones/2019 - Funciones.pdf,png/png_Mates II_2019 - Funciones/asignatura_d...
1333,mates_ccss_probabilidad_2016_reserva_2_ejercic...,Mates CCSS,Mates CCSS,2016,Reserva 2,"Ejercicio 3, Opcion B",Probabilidad,SOCIALES II. 2016. RESERVA 2. EJERCICIO 3. OPC...,De los sucesos A y B de un experimento aleator...,Mates CCSS - Temas/probabilidad/2016 - Probabi...,png/png_Mates CCSS_2016 - Probabilidad/Mates C...
2398,mates_ii_integrales_2019_reserva_2_ejercicio_2...,Mates II,Mates II,2019,Reserva 2,"Ejercicio 2, opción A",Integrales,MATEMÁTICAS II. 2019. RESERVA 2. EJERCICIO 2. ...,x 4\nSea f la función definida por f(x) par a...,Mates II - Temas/integrales/2019 - Integrales.pdf,png/png_Mates II_2019 - Integrales/asignatura_...
3067,quimica_acido_base_2015_septiembre_ejercicio_5...,Química,Química,2015,Septiembre,"Ejercicio 5, opción B",Ácido Base,QUÍMICA. 2015. SEPTIEMBRE. EJERCICIO 5. OPCIÓN B,a) ¿Qué volumen de HCl del 36% en peso y de de...,Química - Temas/ácido_base/2015 - Ácido Bas...,png/png_Química_2015 - Ácido Base/Química_á...


In [28]:
df.isna().sum()

id                    0
asignatura            0
asignatura_carpeta    0
año                   0
convocatoria          0
ejercicio             0
tema                  0
info_ejercicio        0
enunciado             4
archivo               0
imagen                0
dtype: int64

In [29]:
df[df.enunciado.isna()]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
442,fisica_ondas_2016_reserva_3_ejercicio_4_opcion_a,Física,Física,2016,Reserva 3,"Ejercicio 4, opción A",Ondas,FISICA. 2016. RESERVA 3. EJERCICIO 4. OPCIÓN A,NaN,Física - Temas/movimiento_ondulatorio/2016 - ...,png/png_Física_2016 - Ondas/Física_movimiento...
4204,quimica_reacciones_redox_2026_junio_ejercicio_5,Química,Química,2026,Junio,Ejercicio 5,Reacciones Redox,QUÍMICA. 2026. JUNIO. EJERCICIO 5,NaN,Química - Temas/redox/2026 - Reacciones Redox...,png/png_Química_2026 - Reacciones Redox/Quími...
4652,quimica_termoquimica_2025_julio_ejercicio_5,Química,Química,2025,Julio,Ejercicio 5,Termoquímica,QUÍMICA.2025. JULIO. EJERCICIO 5,NaN,Química - Temas/termoquímica/2025 - Termoqui...,png/png_Química_2025 - Termoquímica/Química_...
4655,quimica_termoquimica_2025_reserva_2_ejercicio_5,Química,Química,2025,Reserva 2,Ejercicio 5,Termoquímica,QUÍMICA.2025. RESERVA 2. EJERCICIO 5,NaN,Química - Temas/termoquímica/2025 - Termoqui...,png/png_Química_2025 - Termoquímica/Química_...


In [30]:
df.asignatura.unique()

<StringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [31]:
df.loc[:, ["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,Campo eléctrico y magnético
121,Física,Campo gravitatorio
244,Física,Física cuántica y nuclear
362,Física,Ondas
449,Física,Óptica geométrica
486,Mates CCSS,Contraste De Hipótesis
545,Mates CCSS,Distribución Binomial
550,Mates CCSS,Funciones CCSS
806,Mates CCSS,Inferencia Estadística
1078,Mates CCSS,Matrices Y Determinantes


In [20]:
df.tema = df.tema.str.strip()

In [32]:
df.loc[df.tema == "Matrices Y Determinantes", "tema"] = "Matrices y determinantes"

In [33]:
df[df.asignatura == "Mates CCSS"].tema.unique()

<StringArray>
[        'Contraste De Hipótesis',          'Distribución Binomial',
                  'Funciones CCSS',         'Inferencia Estadística',
        'Matrices y determinantes',                    'Probabilidad',
            'Programación Lineal', 'Sistemas de Ecuaciones Lineales']
Length: 8, dtype: str

In [22]:
df.convocatoria.unique()

<StringArray>
[             'Julio',              'Junio',          'Reserva 1',
          'Reserva 2',          'Reserva 3',          'Reserva 4',
         'Septiembre', 'Reserva1 Ejercicio',           'Ponencia',
             'Modelo']
Length: 10, dtype: str

In [23]:
df[df.convocatoria == "Reserva1 Ejercicio"]

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
195,fisica_campo_gravitatorio_2020_reserva1_ejerci...,Física,Física,2020,Reserva1 Ejercicio,1,Campo gravitatorio,FISICA. 2020. RESERVA1. EJERCICIO 1,a) Considere dos partículas de igual masa sepa...,Física - Temas/campo_gravitatorio/2020 - Camp...,png/png_Física_2020 - Campo gravitatorio/Físi...


In [24]:
df.loc[223, ["convocatoria", "ejercicio"]] = ("Reserva 1", "Ejercicio 1")

In [37]:
df.to_csv(root/"enunciados_ejercicios/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")

In [38]:
df = pd.read_csv(root / "enunciados_ejercicios/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [39]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
2461,mates_ii_integrales_2014_reserva_3_ejercicio_2...,Mates II,Mates II,2014,Reserva 3,"Ejercicio 2, opción B",Integrales,MATEMÁTICAS II. 2014. RESERVA 3. EJERCICIO 2. ...,"x9\nSea f :(1,3) la función definida por f(...",Mates II - Temas/integrales/2014 - Integrales.pdf,png/png_Mates II_2014 - Integrales/asignatura_...
1918,mates_ii_funciones_2008_reserva_3_ejercicio_1_...,Mates II,Mates II,2008,Reserva 3,"Ejercicio 1, opción A",Funciones,MATEMÁTICAS II. 2008. RESERVA 3. EJERCICIO 1. ...,"S ea f :0,2 la función definida por f(x)e...",Mates II - Temas/funciones/2008 - Funciones.pdf,png/png_Mates II_2008 - Funciones/asignatura_d...
2374,mates_ii_integrales_2021_reserva_1_ejercicio_3,Mates II,Mates II,2021,Reserva 1,Ejercicio 3,Integrales,MATEMÁTICAS II. 2021. RESERVA 1. EJERCICIO 3,Considera la función f :  definida por f(x)e...,Mates II - Temas/integrales/2021 - Integrales.pdf,png/png_Mates II_2021 - Integrales/asignatura_...
